# Quick Start

This notebook trains a minimal FSM-LVSM model on a **single scene**, repeated across all steps. 

This should run on a single A6000 machine and finish in under half an hour.

It is intended for smoke-testing your setup, debugging model architecture, and validating the training loop, not for producing a generalizable model.

## Prerequisites

Run this cell first. If any import fails, ensure your virtual environment is activated and dependencies are installed.

In [ ]:
import os
import yaml
import json
from pathlib import Path
from omegaconf import OmegaConf

import math
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

These import from the local utility files. Make sure you are running this notebook from the **project root** (where `fsm/` and `utils_train.py` live), otherwise these will fail with `ModuleNotFoundError`.

In [ ]:
from fsm.model.losses.ewc_loss import ewc_training_loss
from fsm.model.metrics import save_random_samples_with_psnr
from fsm.data.dataset_fromvid import NVSVideoDataset
from utils_train import (
    get_class_by_name,
    get_optimizer,
    get_lr_scheduler,
    get_ewc_config,
    save_checkpoint,
    discover_fast_blocks,
    init_ewc_buffers,
    maybe_reanchor,
    build_info
)

`every(n, step)` returns `True` when `step` is a multiple of `n`, or within the first `also_first` steps, used to gate logging, visualization, and checkpointing.

In [ ]:
def every(n, step, also_first=10):
    return (step % n == 0) or (step <= also_first)

## Training Configurations

Defaults to CUDA if available, CPU otherwise. Training on CPU is possible but very slow, a GPU is strongly recommended.

In [ ]:
# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 42
torch.manual_seed(seed)

We will use the provided toy config and experiment name throughout this notebook. Edit these if you want to point to a different setup.

In [ ]:
# ── Hardcoded config ────────────────────────
EXP_NAME      = "fsm_quickstart"
CONFIG_PATH   = "fsm/configs/train/fsm_lvsm_toy.yaml"
config = OmegaConf.load(CONFIG_PATH)

For this quickstart, we use a lightweight config:

| Parameter | Value |
|---|---|
| Patch size | 8 |
| Dimension | 768 |
| Layers | 8 |
| Image size | 128 × 128 |
| Max frames | 128 |

In [ ]:
display(yaml.safe_load(OmegaConf.to_yaml(config.model)))

Instantiates the model class specified in the config.
```yaml
model:
  class_name: fsm.model.model_4dlvsm.FSM4DLVSM
```

On first run, this will trigger an automatic download of the LPIPS network weights, expect a short delay.

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
FSM   = get_class_by_name(getattr(config.model, "class_name"))
model = FSM(config).to(device)

Inspect the training configs.

| Parameter | Value | Description |
|---|---|---|
| `bs_per_gpu` | 2 | Batch size |
| `steps` | 2500 | Total training steps |
| `warmup_steps` | 500 | LR warmup; LPIPS also activates after this |
| `num_input_views` | 12 | Context views fed to the model |
| `num_target_views` | 12 | Views the model is asked to render |
| `lr` | 1e-4 | Peak learning rate |

In [ ]:
display(yaml.safe_load(OmegaConf.to_yaml(config.training)))

AdamW with cosine decay and a linear warmup over the first `warmup_steps` iterations. All values are read from the config.

In [ ]:
# ── Optimizer & scheduler ─────────────────────────────────────────────────────
optimizer = get_optimizer(
    model,
    weight_decay=config.training.weight_decay,
    learning_rate=config.training.lr,
    betas=config.training.betas
)

lr_scheduler = get_lr_scheduler(
    optimizer,
    warm_up_steps=config.training.warmup_steps,
    total_train_steps=config.training.steps,
    scheduler_type="cosine"
)

Automatically resumes from the latest checkpoint in `outputs/<EXP_NAME>/` if one exists, safe to re-run this cell. If no checkpoint is found, training starts from scratch.

Note `reset_optim: True` in the config means optimizer and scheduler states are **not** restored on resume, only model weights are. Change this to `False` to do a full resume.

In [ ]:
# ── Checkpoint restore ────────────────────────────────────────────────────────
now_iters   = 0
output_dir  = f"outputs/{EXP_NAME}"
os.makedirs(output_dir, exist_ok=True)
ewc_buffers = None   # created lazily after model is built

if os.path.isdir(output_dir):
    checkpoints = [f for f in os.listdir(output_dir) if f.startswith("model_") and f.endswith(".pth")]

    if checkpoints:
        latest_checkpoint = max(checkpoints, key=lambda x: int(x.split("_")[1].split(".")[0]))
        checkpoint_path   = os.path.join(output_dir, latest_checkpoint)
        print(f"Loading checkpoint from {checkpoint_path}...")

        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        model.load_state_dict(checkpoint["model"])

        if not getattr(config.training, "reset_optim", False):
            print("Restoring optimizer and lr_scheduler states...")
            optimizer.load_state_dict(checkpoint["optimizer"])
            lr_scheduler.load_state_dict(checkpoint["lr_scheduler"])
            now_iters = checkpoint.get("now_iters", 0)

        if "ewc_buffers" in checkpoint:
            ewc_buffers = {
                k: {kk: vv.to(device=device) for kk, vv in v.items()}
                for k, v in checkpoint["ewc_buffers"].items()
            }
            print("Restored EWC buffers from checkpoint.")

print(f"Starting from iter {now_iters}.")

## Dataset

We use a single example scene from Stereo4D for this overfitting demo. 

The dataset config is minimal: one scene, repeated across all training steps.

In [ ]:
display(yaml.safe_load(OmegaConf.to_yaml(config.dataset)))

A lightweight wrapper around `NVSVideoDataset` that loads a single scene once and repeats it. The scene is cached in memory after the first load, subsequent iterations have zero I/O overhead. `toy_length` controls how many samples constitute one epoch. With `toy_length=1000`.

In [ ]:
class ToyRepeatDataset(Dataset):
    """
    Loads ONE scene from NVSVideoDataset and repeats it `length` times.
    Useful for smoke-testing the training loop without a full dataset.
    """

    def __init__(self, single_scene_dataset: NVSVideoDataset, length: int = 1000):
        self._ds     = single_scene_dataset
        self._length = length
        self._cache  = None   # lazy: load once on first access

    def __len__(self):
        return self._length

    def __getitem__(self, index):
        if self._cache is None:
            self._cache = self._ds[0]
        return self._cache


def get_dataset(
    num_views,
    img_size,
    num_input_views,
    max_frame_time  = 128,
    scene_pose_normalize = True,
    window_size     = 128,
    sorted_indices  = True,
    toy_data_path   = "static/datasets/example/example.json",
    toy_length      = 1000,
):
    """
    Returns a ToyRepeatDataset that yields the same single scene repeatedly,
    and None for the sampler (DataLoader will use shuffle=True instead).
    """
    base_dataset = NVSVideoDataset(
        data_path           = toy_data_path,
        num_views           = num_views,
        image_size          = img_size,
        sorted_indices      = sorted_indices,
        scene_pose_normalize= scene_pose_normalize,
        max_frame_time      = max_frame_time,
        num_input_views     = num_input_views,
        window_size         = window_size,
    )

    dataset = ToyRepeatDataset(base_dataset, length=toy_length)
    sampler = None   # single-node: DataLoader handles shuffling

    print(f"[ToyDataset] Loaded 1 scene from: {toy_data_path}")
    print(f"[ToyDataset] Repeating it {toy_length} times per epoch.")

    return dataset, sampler

Reads view counts and resolution from config. For this toy run: **24 total views** (12 input + 12 target) at **128 × 128** resolution.

In [ ]:
# ── Dataset & dataloader ──────────────────────────────────────────────────────
num_all_views    = config.training.num_all_views
num_input_views  = config.training.num_input_views
num_target_views = config.training.num_target_views

resolution = config.model.image_tokenizer.image_size
img_size   = (resolution, resolution) if isinstance(resolution, int) else tuple(resolution)

Inspect the toy data path, you should see `['static/datasets/example/fZcNop_Aenw_88573338.jsonl']`.

In [ ]:
toy_data_path = "static/datasets/example/example.json"
with open(toy_data_path) as f:
    scene_list = json.load(f)
print(scene_list)

Inspect this scene, you should see:
```bash
dict_keys(['scene_name', 'video_path', 'num_video_frames', 'frames'])
fZcNop_Aenw_88573338
static/datasets/example/fZcNop_Aenw_88573338-left_rectified.mp4
199
dict_keys(['frame_idx', 'w2c', 'fx', 'fy', 'cx', 'cy', 'h', 'w'])
```

In [ ]:
with open(scene_list[0]) as f:
    scene_info = json.load(f)
print(scene_info.keys())
print(scene_info['scene_name'])
print(scene_info['video_path'])
print(scene_info['num_video_frames'])
print(scene_info['frames'][0].keys())

Preview the video.

In [ ]:
from IPython.display import Video
Video(scene_info['video_path'], embed=True, width=640)

Instantiate the dataset and dataloader. Note a few notebook-friendly overrides from the original training script:

| Parameter | Notebook value | Reason |
|---|---|---|
| `num_workers` | 0 | Avoids multiprocessing issues in Jupyter |
| `persistent_workers` | False | Required when `num_workers=0` |
| `prefetch_factor` | None | Required when `num_workers=0` |
| `drop_last` | False | Only one scene, keep all batches |

In [ ]:
dataset, sampler = get_dataset(
    num_views=num_all_views,
    img_size=img_size,
    num_input_views=num_input_views,
    max_frame_time=config.model.get("max_frames", 256),
    scene_pose_normalize=config.dataset.get("scene_pose_normalize", True),
    window_size=config.dataset.get("window_size", 256),
    sorted_indices=config.dataset.get("sort_by_time", True),
    toy_data_path=toy_data_path,
    toy_length=1000
)

dataloader_seed_generator = torch.Generator()
dataloader_seed_generator.manual_seed(seed)

dataloader = DataLoader(
    dataset,
    batch_size=config.training.bs_per_gpu,
    shuffle=(sampler is None),
    sampler=sampler,
    num_workers=0,
    persistent_workers=False,
    pin_memory=config.training.get("pin_memory", True),
    drop_last=False,
    prefetch_factor=None,
    generator=dataloader_seed_generator
)

## Model Training

Initializes Elastic Weight Consolidation (EWC) buffers for the model's fast-weight blocks (LaCT layers). 

These buffers track parameter importance to regularize fast-weight updates during training.

In [ ]:
# ── EWC config ────────────────────────────────────────────────────────────────
ewc_config      = get_ewc_config(config)
ewc_enable      = ewc_config["enable"]
ewc_lambda_prox = ewc_config["lambda_prox"]
ewc_alpha       = ewc_config["alpha"]
ewc_mode        = ewc_config["mode"]
ewc_anchor      = ewc_config["anchor_policy"]
ewc_anchor_beta = ewc_config["anchor_beta"]
ewc_loss_weight = ewc_config["lambda_train"]

fast_blocks = discover_fast_blocks(model)
if ewc_enable and (ewc_buffers is None):
    ewc_buffers = init_ewc_buffers(fast_blocks, device=device)

anchor_mode = 1 if "streaming" in ewc_anchor else 0
if "ema" in ewc_anchor:
    anchor_mode *= 2
for _, blk in fast_blocks:
    setattr(blk, "anchor_mode", anchor_mode)
    if "ema" in ewc_anchor:
        setattr(blk, "anchor_beta", ewc_anchor_beta)

fsm_fast_blocks = discover_fast_blocks(model)

Run the training loop.

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
remaining_steps = config.training.steps - now_iters
print(f"Starting training for {remaining_steps} steps...")

for epoch in range((remaining_steps - 1) // len(dataloader) + 1):
    if sampler is not None:
        dataloader.sampler.set_epoch(epoch)

    if ewc_enable and "online" in ewc_anchor:
        maybe_reanchor(fast_blocks, ewc_buffers, ewc_anchor)

    for data_dict in dataloader:
        data_dict        = {k: v.to(device) for k, v in data_dict.items() if isinstance(v, torch.Tensor)}
        input_data_dict  = {k: v[:, :num_input_views]  for k, v in data_dict.items()}
        target_data_dict = {k: v[:, -num_target_views:] for k, v in data_dict.items()}

        batch_size   = next(iter(data_dict.values())).shape[0]
        fsm_info_dict = build_info(
            fsm_fast_blocks, ewc_buffers, batch_size=batch_size,
            ewc_enable=ewc_enable, lambda_prox=ewc_lambda_prox,
            alpha=ewc_alpha, mode=ewc_mode
        )

        model.set_curriculum(now_iters)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(dtype=torch.bfloat16, device_type="cuda", enabled=(device.type == "cuda")):
            rendering, losses = model(input_data_dict, target_data_dict, info=fsm_info_dict)
            loss = losses.loss

            if ewc_enable and ewc_loss_weight > 0.0 and fast_blocks:
                ewc_loss_train = ewc_training_loss(fast_blocks, ewc_buffers)
                loss = loss + ewc_loss_weight * ewc_loss_train
            else:
                ewc_loss_train = torch.tensor(0.0, device=device)

        loss.backward()

        # Gradient clipping
        skip_optimizer_step = False
        if now_iters <= config.training.get("grad_clip_warmup_steps", 1000):
            global_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float("inf"))
        else:
            global_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config.training.get("grad_clip_norm", 1.0))
            if not math.isfinite(global_grad_norm):
                skip_optimizer_step = True
            elif global_grad_norm > config.training.get("grad_threshold", 4.0):
                skip_optimizer_step = True

        if not skip_optimizer_step:
            optimizer.step()

        lr_scheduler.step()
        now_iters += 1

        # ── Logging ───────────────────────────────────────────────────────────
        if every(config.training.log_every, now_iters):
            model.eval()
            with torch.no_grad():
                parts = [f"Iter {now_iters:07d}"]
                if losses.loss            is not None: parts.append(f"Loss {losses.loss.item():.4f}")
                if losses.psnr            is not None: parts.append(f"PSNR {losses.psnr:.2f}")
                if losses.lpips_loss      is not None: parts.append(f"LPIPS {losses.lpips_loss.item():.4f}")
                if losses.l2_loss         is not None: parts.append(f"L2 {losses.l2_loss.item():.4f}")
                if losses.perceptual_loss is not None: parts.append(f"Perceptual {losses.perceptual_loss.item():.4f}")
                if losses.ssim_loss       is not None: parts.append(f"SSIM {losses.ssim_loss.item():.4f}")
                if losses.pixelalign_loss is not None: parts.append(f"PixelAlign {losses.pixelalign_loss.item():.4f}")
                if losses.pointsdist_loss is not None: parts.append(f"PointsDist {losses.pointsdist_loss.item():.4f}")
                print(" | ".join(parts) + " |")

                try:
                    if every(config.training.visualize_every, now_iters):
                        save_random_samples_with_psnr(
                            rendering=rendering,
                            target=target_data_dict["image"],
                            now_iters=now_iters,
                            output_dir=output_dir,
                            sample_n=3,
                            wandb_run=None,
                            title_prefix="render_vs_gt_with_psnr"
                        )
                except Exception as e:
                    print(f"[warn] sample logging failed at iter {now_iters}: {e}")

            model.train()

        # ── Checkpointing ─────────────────────────────────────────────────────
        if every(config.training.save_every, now_iters, also_first=-1):
            save_checkpoint(
                Path(output_dir) / f"model_{now_iters:07d}.pth",
                model, optimizer, lr_scheduler, now_iters, epoch, ewc_buffers
            )

        if now_iters == config.training.steps:
            break

print("Training complete.")

## Expected Results

By step 2500, the model should be cleanly overfitting the single scene:
```
Iter 0002500 | Loss 0.0039 | PSNR 33.78 | LPIPS 0.0070 | L2 0.0004 |
```

Checkpoints and visualizations are saved to `outputs/fsm_quickstart/`:
```
outputs/
└── fsm_quickstart/
    ├── 1/
    │   └── render_vs_gt_with_psnr.png
    ├── ...
    ├── 2500/
    │   └── render_vs_gt_with_psnr.png
    ├── model_0000500.pth
    ├── model_0001000.pth
    ├── model_0001500.pth
    ├── model_0002000.pth
    └── model_0002500.pth
```

Reconstruction quality at step 2500:

![quickstart_training_step2500](static/imgs/quickstart_training_step2500.png)